In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.insert(1, '../')

In [2]:
import pyFBS
from pyFBS.utility import *
import pickle
import pyvista as pv

In [3]:
import numpy as np 
import pyvista as pv
from numpy import cross, eye
from scipy.linalg import expm, norm
import sys

In [4]:
import pyvista as pv
import numpy as np
import numpy as np

import cv2

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
import keyboard as kb

In [7]:

def isRotationMatrix(R):
    """"
    Check if a matrix is a valid rotation matrix.
    """
    Rt = np.transpose(R)
    _ident = np.dot(Rt, R)
    I = np.identity(3, dtype = R.dtype)
    n = np.linalg.norm(I - _ident)
    return n < 1e-6


def rotationMatrixToEulerAngles(R) :
    """
    Calculates rotation matrix to euler angles
    """
    assert(isRotationMatrix(R))
    
    sy = math.sqrt(R[0,0] * R[0,0] +  R[1,0] * R[1,0])
    
    singular = sy < 1e-6

    if  not singular :
        x = math.atan2(R[2,1] , R[2,2])
        y = math.atan2(-R[2,0], sy)
        z = math.atan2(R[1,0], R[0,0])
    else :
        x = math.atan2(-R[1,2], R[1,1])
        y = math.atan2(-R[2,0], sy)
        z = 0
    
    eps = 1e-8
    array = np.array([x, y, z])
    return array

def M(axis, theta):
    """
    Euler-Rodrigues formula
    """
    t = expm(cross(eye(3), axis/norm(axis)*(theta)))
    #print(theta)
     
    return t

    
#Create a unit vector
def unit_vector(vec):
    unit = vec / np.linalg.norm(vec)
    return unit


# Find the angle between two unit vector
def angle(vector1, vector2):
    """ 
    Returns the angle in radians between given vectors
    """
    v1_u = unit_vector(vector1)
    v2_u = unit_vector(vector2)
    minor = np.linalg.det(
        np.stack((v1_u[-2:], v2_u[-2:]))
    )
    if minor == 0:
        sign = 1
    else:
        sign = -np.sign(minor)
    dot_p = np.dot(v1_u, v2_u)
    dot_p = min(max(dot_p, -1.0), 1.0)
    return sign * np.arccos(dot_p)


def rotation_matrix_from_vectors(vec1, vec2):
    """ Find the rotation matrix that aligns vec1 to vec2
    :param vec1: A 3d "source" vector
    :param vec2: A 3d "destination" vector
    :return mat: A transform matrix (3x3) which when applied to vec1, aligns it with vec2.
    """
    vec1 += np.random.random(3)/1e10 # just to avoid possible math errors
    vec2 += np.random.random(3)/1e10 # just to avoid possible math errors
    
    a, b = (vec1 / np.linalg.norm(vec1)).reshape(3), (vec2 / np.linalg.norm(vec2)).reshape(3)


    if (np.abs(a) == np.abs(b)).all():
        return np.diag([1,1,1])
    else:
        v = np.cross(a, b)
        c = np.dot(a, b)
        s = np.linalg.norm(v)
        kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        rotation_matrix = np.eye(3) + kmat + kmat.dot(kmat) * ((1 - c) / (s ** 2))

        return rotation_matrix

def unit_vector(vector):
    """ 
    Returns the unit vector of the vector.  
    """
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    """ 
    Returns the angle in radians between vectors 'v1' and 'v2'
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))

In [8]:
class DynamicPosition():
    def __init__(self,objects,p,N,mesh = None,snap_outward = True,size = 1):
        
        self.size = size
        self.points = np.array([[size/2,size/2,size/2],
                   [0,size/2,size/2],
                   [size/2,0,size/2],
                   [size/2,size/2,0]])

        
        # Creates a bounding box
        self.box = pv.Box((-size,size,-size,size,-size,size))
        self.box.translate([size,size,size])
        self.box.points /= 2
        
        
        # defines the objects
        objects.insert(0, self.box)
        self.objects = objects
        
        # get the number of dynamic stuff in the display window
        self.N = int(N*4)
    
        # local orientation of the bounding box/object
        self.local_orientation = np.asarray([[1, 0, 0],
                          [0, 1, 0],
                          [0, 0, 1]])

        # local positions of the point widgets
        self.local_widgets = np.array([[0.0, 0.0, 0.0],
                   [-0.5, 0, 0],
                   [0, -0.5, 0],
                   [0, 0, -0.5]])*size
        
        # local normals on which the snapping happens
        self.local_normals = np.asarray([[1, 0, 0],
                                        [0, 1, 0],
                                        [0, 0, 1],
                                        [-1, 0, 0],
                                        [0, -1, 0],
                                        [0, 0, -1]]).T
        
        # ray_size on which the snapping to the mesh happens
        ray_size = 4*size
        self.local_rays = np.asarray([[1, 0, 0],
                                        [0, 1, 0],
                                        [0, 0, 1],
                                        [-1, 0, 0],
                                        [0, -1, 0],
                                        [0, 0, -1]]).T * ray_size
        
        # computes mesh normals
        self.mesh = mesh
        self.mesh.compute_normals(auto_orient_normals = True,inplace = True)
        
        # disables the rotation of the object
        self.turn_on = False
        
        self.snap_outward = snap_outward
        
        
    def get_pos_orient(self,euler_angles = False,one_dir = None,eps = 1e-5):
        position = self.box.center_of_mass()
        
        if euler_angles:
            orientation = rotationMatrixToEulerAngles(self.local_orientation)
        elif one_dir != None:
            orientation = self.local_orientation[:,one_dir]
        else:
            orientation = self.local_orientation
            
        orientation[np.abs(orientation) < eps] = 0

        return position,orientation
        
        
    def translate(self,point,snap = False):
        # definest rays to find intersection with the supplied mesh
        point1x = point + self.local_rays[:,0]
        point2x = point + self.local_rays[:,3]
        
        point1y = point + self.local_rays[:,1]
        point2y = point + self.local_rays[:,4]
        
        point1z = point + self.local_rays[:,2]
        point2z = point + self.local_rays[:,5]
        
        
        # performs ray trace in three direction        
        points_x, ind_x = self.mesh.ray_trace(point1x,point2x)
        points_y, ind_y = self.mesh.ray_trace(point1y,point2y)
        points_z, ind_z = self.mesh.ray_trace(point1z,point2z)
        
        # stacks all the ray intersections
        points = np.vstack([points_x,points_y,points_z])
        ind = np.hstack([ind_x,ind_y,ind_z])

        # default option - no rotation
        rot = np.diag([1,1,1])
        
        # if there is an intersection and if "t" is not pressed go forward
        if points.size != 0 and not(kb.is_pressed('t')):        
            list_ind = []
            # go through all the intersections
            for i in range(len(points)):
                _point = points[i]
                p1 = _point
                p2 = point
                gg  = np.sqrt( ((p1[0]-p2[0])**2)+((p1[1]-p2[1])**2)+((p1[2]-p2[2])**2) )
                
                list_ind.append(gg)
            
            # find the closest to the box center
            _sel = np.argmin(list_ind)
            
            # find the nearest normal
            v2 = self.mesh.cell_normals[int(ind[_sel])]
            th = []
            for _loc in self.local_normals.T:
                th.append(angle_between(_loc, v2))
            closest_orient = self.local_normals.T[np.argmin(th)]
            
            # find orientation between box orientation and cell normal 
            f = self.mesh.cell_normals[int(ind[_sel])] 
            t = closest_orient + np.random.random(3)/1e20 
            
            # push box 0.5 away from the normal
            if self.snap_outward:
                point = points[_sel] + f/2*self.size  
            else:
                point = points[_sel]
                
            # define rotational matrix to allign with the surface normal
            rot = rotation_matrix_from_vectors(t,f)
        
        # move everything to a new location
        _new = point - self.box.center_of_mass() 
        
        # snaps to the mesh and moves point widgets to the new location
        if snap:
            # translates 
            for item in self.objects:
                item.translate(_new)  
            
            p.sphere_widgets[self.N+0].SetCenter(point)
            t_new = self.box.center_of_mass()

            for k in range(3):    
                self.local_widgets[k+1, :] = rot@self.local_widgets[k+1, :]
                p.sphere_widgets[self.N +  k+1].SetCenter(self.local_widgets[k+1, :]  + t_new)

            # orient the local csys of accelerometer with the new rotation
            self.local_orientation = (rot @ (self.local_orientation).T).T 
            self.local_normals = rot @ self.local_normals
            self.local_rays = rot @ self.local_rays
            
            #rotate everything within accelerometer
            for item in self.objects:
                item.points = (rot @ (item.points - t_new).T).T + t_new 
            
        else:
            for item in self.objects:
                item.translate(_new)
            for i in range(4):
                p.sphere_widgets[self.N  + i].SetCenter(_new + np.asarray(p.sphere_widgets[self.N + i].GetCenter()))
        
        
        
    def callback(self,point, i):
        # 3D translation in space
        if i == 0:
            self.translate(point,snap = True)
        
        # 3D rotation in space
        else:
            if self.turn_on:
                # get the center of acc
                _new = self.box.center_of_mass()
                _vec1 = np.asarray(point-_new)
                _vec2 = (np.asarray(self.local_widgets[i,:])) 

                # define the rotational matrix based on angle of rotation
                theta = angle(_vec1, _vec2)
                rot = M(self.local_orientation[i-1, :], theta)

                # rotate everything within accelerometer
                for item in self.objects:
                    item.points = (rot @ (item.points - _new).T).T + _new 

                # orient the local csys of accelerometer with the new rotation
                self.local_orientation = (rot @ (self.local_orientation).T).T 
                self.local_normals = rot @ self.local_normals
                self.local_rays = rot @ self.local_rays

                # position all widgets to the new position
                for k in range(4):
                    self.local_widgets[k, :] = rot@self.local_widgets[k, :]
                    p.sphere_widgets[self.N +  k].SetCenter(self.local_widgets[k, :]  + _new)

In [9]:
view3D = pyFBS.view3D(show_origin = False)

In [10]:
stl = pyFBS.example_lab_testbench["STL"]["A"]
#stl = pyFBS.example_auto_testbench["STL"]["receiver"]

mesh = pv.PolyData(stl)
#mesh.points /= 10

view3D.plot.add_mesh(mesh,name = "ts",color = "#D3D3D3")
#view3D.plot.add_mesh(mesh,name = "ts_p",color = "b",style = "points")

(vtkRenderingOpenGL2Python.vtkOpenGLActor)000001EADE6F78E8

In [11]:
# Add accelerometer
p = view3D.plot

view3D.all_accs = []
def add_accelerometer(point):
    try:
        i = int(len(view3D.plot.sphere_widgets)/4)
    except:
        i = 0
        
    acc = view3D.create_accelerometer([5,5,5],[0,0,0],size = 10)
    view3D.add_accelerometer(acc)
    
    _gg = DynamicPosition(acc,view3D.plot,i,mesh = mesh,size = 10)
    view3D.plot.add_sphere_widget(_gg.callback, center=_gg.points, color = ["k","r","g","b"],radius = 10/15)
    
    _gg.turn_on = True
    _gg.translate(point)
    
    view3D.all_accs.append(_gg)

view3D.plot.enable_point_picking(callback = add_accelerometer,color = "r",show_message="",show_point = False)
view3D.plot.add_text("Press P too add an accelerometer (hold down letter T to disable snapping to mesh).", font_size = 10,color = "k",font  = "times",name = "text")

(vtkRenderingAnnotationPython.vtkCornerAnnotation)000001EADE6F7768

In [12]:
view3D.plot.enable_parallel_projection()

In [15]:
columns_chann = ["Name","Description","Type","DirectionLabel","Quantity","Unit","Component","NodeNumber","Grouping","Position_1","Position_2","Position_3","Orientation_1","Orientation_2","Orientation_3"]
df = pd.DataFrame(columns = columns_chann)

for i,_acc in enumerate(view3D.all_accs):
    pos,euler_dir = _acc.get_pos_orient(euler_angles = True)
    
    data_chn = np.asarray([["Sensor "+ str(1+i),None,None,None,None,None,None,None,None,pos[0],pos[1],pos[2],euler_dir[0]*180/np.pi,euler_dir[1]*180/np.pi,euler_dir[2]*180/np.pi]])
    
    df_row = pd.DataFrame(data = data_chn,columns = columns_chann)
    df = df.append(df_row,ignore_index = True)
df

,Name,Description,Type,DirectionLabel,Quantity,Unit,Component,NodeNumber,Grouping,Position_1,Position_2,Position_3,Orientation_1,Orientation_2,Orientation_3
0,Sensor 1,None,None,None,None,None,None,None,None,-117.281,123.124,37.2064,0,0,7.01785
1,Sensor 2,None,None,None,None,None,None,None,None,-120.99,160.956,32.3538,0,0,0
2,Sensor 3,None,None,None,None,None,None,None,None,-245.92,216.206,-0.262168,0,0,0
3,Sensor 4,None,None,None,None,None,None,None,None,-159.258,251.589,-45.1521,0,0,0
4,Sensor 5,None,None,None,None,None,None,None,None,-225.103,173.982,35.4152,0,0,0
5,Sensor 6,None,None,None,None,None,None,None,None,15.457,219.624,65.6416,0,0,0
6,Sensor 7,None,None,None,None,None,None,None,None,-10.7194,180.457,97.1545,0,0,0


In [16]:
# Add impact
p = view3D.plot

view3D.all_imp = []

def add_impact(point):
    try:
        i = int(len(view3D.plot.sphere_widgets)/4)
    except:
        i = 0
        
    acc,_ = view3D.add_impact([5,5,5],[0,0,1],size = 10)
    
    _gg = DynamicPosition([acc],view3D.plot,i,mesh = mesh,size = 10,snap_outward = False)
    view3D.plot.add_sphere_widget(_gg.callback, center=_gg.points, color = ["k","r","g","b"],radius = 10/15)
    
    _gg.turn_on = True
    _gg.translate(point)
    
    view3D.all_imp.append(_gg)


view3D.plot.enable_point_picking(callback = add_impact,show_message="",show_point = False)
view3D.plot.add_text("Press P too add an impact (hold down letter t to disable snapping to mesh).", font_size = 10,color = "k",font  = "times",name = "text")

(vtkRenderingAnnotationPython.vtkCornerAnnotation)000001EADE6F7A08

In [17]:
columns_chann = ["Name","Description","Type","DirectionLabel","Quantity","Unit","Component","NodeNumber","Grouping","Position_1","Position_2","Position_3","Direction_1","Direction_2","Direction_3"]
df = pd.DataFrame(columns = columns_chann)

for i,_imp in enumerate(view3D.all_imp):
    pos,_dir = _imp.get_pos_orient(one_dir = 2)
    
    data_chn = np.asarray([["Impact "+ str(1+i),None,None,None,None,None,None,None,None,pos[0],pos[1],pos[2],_dir[0],_dir[1],_dir[2]]])
    
    df_row = pd.DataFrame(data = data_chn,columns = columns_chann)
    df = df.append(df_row,ignore_index = True)
df

,Name,Description,Type,DirectionLabel,Quantity,Unit,Component,NodeNumber,Grouping,Position_1,Position_2,Position_3,Direction_1,Direction_2,Direction_3
0,Impact 1,None,None,None,None,None,None,None,None,-169.415,123.852,31.1137,0,0,1
1,Impact 2,None,None,None,None,None,None,None,None,-154.251,120.536,36.5972,0,0,1
2,Impact 3,None,None,None,None,None,None,None,None,-164.977,102.097,42.5479,0,0,1
3,Impact 4,None,None,None,None,None,None,None,None,-190.346,102.766,26.2631,0,0,1
4,Impact 5,None,None,None,None,None,None,None,None,-152.74,142.48,22.4951,0,0,1


In [18]:
# Add VP
p = view3D.plot

view3D.all_vp = []

def add_VP(point):
    try:
        i = int(len(view3D.plot.sphere_widgets)/4)
    except:
        i = 0
        
    acc,_ = view3D.add_vp([2,2,2],size = 4,opacity = .1)
    
    _gg = DynamicPosition([acc],view3D.plot,i,mesh = mesh,size = 4,snap_outward = False)
    view3D.plot.add_sphere_widget(_gg.callback, center=_gg.points, color = ["k","r","g","b"],radius = 4/15)
    
    _gg.turn_on = True
    _gg.translate(point)
    
    view3D.all_vp.append(_gg)


view3D.plot.enable_point_picking(callback = add_VP,show_message="",show_point = False)
view3D.plot.add_text("Press P too add a VP (hold down letter t to disable snapping to mesh).", font_size = 10,color = "k",font  = "times",name = "text")

(vtkRenderingAnnotationPython.vtkCornerAnnotation)000001EADE6F79A8

In [19]:
columns_chann = ["Name","Description","Type","DirectionLabel","Quantity","Unit","Component","NodeNumber","Grouping","Position_1","Position_2","Position_3","Orientation_1","Orientation_2","Orientation_3"]
df = pd.DataFrame(columns = columns_chann)

for i,_acc in enumerate(view3D.all_vp):
    pos,euler_dir = _acc.get_pos_orient(euler_angles = True)
    
    data_chn = np.asarray([["VP "+ str(1+i),None,None,None,None,None,None,None,None,pos[0],pos[1],pos[2],euler_dir[0]*180/np.pi,euler_dir[1]*180/np.pi,euler_dir[2]*180/np.pi]])
    
    df_row = pd.DataFrame(data = data_chn,columns = columns_chann)
    df = df.append(df_row,ignore_index = True)
df

,Name,Description,Type,DirectionLabel,Quantity,Unit,Component,NodeNumber,Grouping,Position_1,Position_2,Position_3,Orientation_1,Orientation_2,Orientation_3
0,VP 1,None,None,None,None,None,None,None,None,-153.056,85.6332,44.4062,0,0,0
1,VP 2,None,None,None,None,None,None,None,None,-169.909,102.736,42.3696,0,0,0
2,VP 3,None,None,None,None,None,None,None,None,-179.514,73.5064,36.1982,0,0,0
